In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1. 異常現象

重複したデータが保存されるようにテーブルを設計すると、`挿入・更新・削除`を行う際に異常現象が発生する。

In [6]:
%%sql

CREATE TABLE employee (
    emp_no      CHAR(4) PRIMARY KEY,
    emp_name    VARCHAR(20),
    address     VARCHAR(100),
    phone       VARCHAR(20),
    dept_no     CHAR(3),
    dept_name   VARCHAR(30),
    dept_loc    VARCHAR(30)
);

++
||
++
++

In [7]:
%%sql

INSERT INTO employee VALUES
('E001', 'ハウル', '東京', '010-1111-1111', 'D01', '営業部', '東京');

INSERT INTO employee VALUES
('E002', '千尋', '横浜', '010-2222-2222', 'D01', '営業部', '東京');

INSERT INTO employee VALUES
('E003', 'キキ', '大阪', '010-3333-3333', 'D02', '開発部', '横浜');

INSERT INTO employee VALUES
('E004', 'パズー', '名古屋', '010-4444-4444', 'D02', '開発部', '横浜');

INSERT INTO employee VALUES
('E005', 'ソフィー', '札幌', '010-5555-5555', 'D03', '人事部', '東京');

++
||
++
++

In [8]:
%%sql

SELECT * FROM employee;

emp_no,emp_name,address,phone,dept_no,dept_name,dept_loc
E001,ハウル,東京,010-1111-1111,D01,営業部,東京
E002,千尋,横浜,010-2222-2222,D01,営業部,東京
E003,キキ,大阪,010-3333-3333,D02,開発部,横浜
E004,パズー,名古屋,010-4444-4444,D02,開発部,横浜
E005,ソフィー,札幌,010-5555-5555,D03,人事部,東京


同じ部署に所属する社員ごとに、`部署番号・部署名・部署所在地`が繰り返し保存されている。

このような重複データによって、次の異常現象が発生する。

- `挿入異常`
- `更新異常`
- `削除異常`

## 1) 更新異常

ある部署の所在地が変更された場合、一部の社員データだけを更新すると、同じ部署内で異なる所在地が保存されてしまう。

例えば、`営業部 （D01）`の所在地が`東京`から`川崎`に変更されたとする。

#### 一部の行だけを更新した場合

In [14]:
%%sql

UPDATE test.employee
SET dept_loc = '川崎'
WHERE emp_no = 'E001';

++
||
++
++

In [15]:
%%sql

SELECT * FROM employee;

emp_no,emp_name,address,phone,dept_no,dept_name,dept_loc
E001,ハウル,東京,010-1111-1111,D01,営業部,川崎
E002,千尋,横浜,010-2222-2222,D01,営業部,東京
E003,キキ,大阪,010-3333-3333,D02,開発部,横浜
E004,パズー,名古屋,010-4444-4444,D02,開発部,横浜
E005,ソフィー,札幌,010-5555-5555,D03,人事部,東京


この場合、同じ`D01`の営業部であるにもかかわらず、所在地が異なる状態になる。

| emp_no | emp_name | dept_no | dept_name | dept_loc |
|---|---|---|---|---|
| E001 | ハウル | D01 | 営業部 | 川崎 |
| E002 | 千尋 | D01 | 営業部 | 東京 |

これは、部署情報が社員ごとに重複して保存されているために発生する`更新異常`である。

#### 同じ部署のすべての行を更新する場合

In [16]:
%%sql

UPDATE test.employee
SET dept_loc = '川崎'
WHERE dept_no = 'D01';

++
||
++
++

In [17]:
%%sql

SELECT * FROM employee;

emp_no,emp_name,address,phone,dept_no,dept_name,dept_loc
E001,ハウル,東京,010-1111-1111,D01,営業部,川崎
E002,千尋,横浜,010-2222-2222,D01,営業部,川崎
E003,キキ,大阪,010-3333-3333,D02,開発部,横浜
E004,パズー,名古屋,010-4444-4444,D02,開発部,横浜
E005,ソフィー,札幌,010-5555-5555,D03,人事部,東京


#### ※ `dept_no`が`D01`であるすべての社員データを更新しなければ、部署情報の整合性を維持できない。

> **更新異常**：重複して保存された同じ情報をすべて更新しなかったため、データの内容が一致しなくなる現象。

## 2) 挿入異常

新しい部署を作成しても、所属する社員がまだいない場合、部署情報だけを登録できない。

例えば、次の部署を登録するとする。

- 部署番号：`D04`
- 部署名：`マーケティング部`
- 所在地：`大阪`

しかし、現在のテーブルは社員情報を基準としているため、次のように社員情報を`NULL`にする必要がある。

In [20]:
%%sql

INSERT INTO test.employee
VALUES
(NULL, NULL, NULL, NULL, 'D04', 'マーケティング部', '大阪');

RuntimeError: (pymysql.err.IntegrityError) (1048, "Column 'emp_no' cannot be null")
[SQL: INSERT INTO test.employee
VALUES
(NULL, NULL, NULL, NULL, 'D04', 'マーケティング部', '大阪');]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


ただし、`emp_no`は`PRIMARY KEY`なので`NULL`を登録できない。

つまり、社員がいなければ部署情報だけを保存できない。これを`挿入異常`という。

> **挿入異常**：ある情報を登録するために、不要なデータや存在しないデータまで入力しなければならない現象。

### 3) 削除異常

ある部署に所属する社員が1人しかいない場合、その社員を削除すると部署情報まで一緒に失われる。

例えば、`E005`が人事部に所属する唯一の社員だとする。

In [21]:
%%sql

DELETE
FROM test.employee
WHERE emp_no = 'E005';

++
||
++
++

In [22]:
%%sql

SELECT * FROM employee;

emp_no,emp_name,address,phone,dept_no,dept_name,dept_loc
E001,ハウル,東京,010-1111-1111,D01,営業部,川崎
E002,千尋,横浜,010-2222-2222,D01,営業部,川崎
E003,キキ,大阪,010-3333-3333,D02,開発部,横浜
E004,パズー,名古屋,010-4444-4444,D02,開発部,横浜


この社員を削除すると、社員情報だけでなく、`D03（人事部）`の部署名や所在地もテーブルから消える。

つまり、部署自体を削除する目的ではないにもかかわらず、部署情報まで失われる。これを`削除異常`という。

> **削除異常**：不要なデータを削除した際に、残すべき別の情報まで一緒に失われる現象。